# Phase 6: RAG (Fine-tuned BGE + Extractive Reranker)

**Pipeline**:
```
NL Query → Fine-tuned BGE embedder → FAISS → Top-20 candidates
                                               ↓
                               Fine-tuned RoBERTa reranker → Best answer
```

**Target: Beat RESTBERTa paper (0.8529 on PM)**

**Setup**:
- Kaggle: GPU **T4 x2**, Internet ON
- Add dataset: `kailram/restberta-datasets`
- Run all cells in order. After Cell 1, restart kernel.

**Total time**: ~5-6 hours (BGE 2hr + Reranker 2hr + Eval 1-2hr)

In [ ]:
# Cell 1: Install (compatible versions)
!pip install -q sentence-transformers==3.3.1 faiss-cpu==1.9.0 "transformers>=4.44.0,<4.48" "huggingface_hub>=0.25,<0.27" accelerate datasets
print('Installed. Now Kernel → Restart Session')

In [ ]:
# Cell 2: Imports + Config
import os, json, gc, glob, time, random, re, shutil
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from datetime import datetime
from tqdm import tqdm

from sentence_transformers import SentenceTransformer, InputExample, losses
from transformers import (
    AutoTokenizer, AutoModelForQuestionAnswering,
    TrainingArguments, Trainer,
)

# Paths
DATA_BASE = '/kaggle/input/datasets/kailram/restberta-datasets'
PM_PATH   = os.path.join(DATA_BASE, 'parameter_matching')
ED_PATH   = os.path.join(DATA_BASE, 'endpoint_discovery')
OUTPUT    = '/kaggle/working'
os.makedirs(f'{OUTPUT}/checkpoints', exist_ok=True)
os.makedirs(f'{OUTPUT}/results', exist_ok=True)

# Models
BGE_CKPT     = 'BAAI/bge-small-en-v1.5'
BGE_NAME     = 'bge_small_restberta'
RERANK_CKPT  = 'FacebookAI/roberta-base'
RERANK_NAME  = 'roberta_reranker'

# Hyperparams
SEED              = 42
BGE_TRAIN_PAIRS   = 100_000
BGE_EPOCHS        = 2
BGE_BATCH         = 64
BGE_LR            = 2e-5

RERANK_TRAIN_QA   = 60_000
RERANK_EPOCHS     = 2
RERANK_BATCH      = 16
RERANK_LR         = 3e-5

TOP_K_RETRIEVE    = 20
MAX_LENGTH        = 512

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

# Verify data
for split in ['parameter_matching/train/train.json',
              'parameter_matching/validation/validation.json',
              'endpoint_discovery/train/train.json',
              'endpoint_discovery/validation/validation.json']:
    p = os.path.join(DATA_BASE, split)
    sz = os.path.getsize(p)/1e6 if os.path.exists(p) else 0
    print(f"  {'OK' if sz else 'MISSING':>7}  {split} ({sz:.0f} MB)")

In [ ]:
# Cell 3: Helper functions
def precompute_props(ctx):
    props = []
    i, n = 0, len(ctx)
    while i < n:
        while i < n and ctx[i] == ' ': i += 1
        if i >= n: break
        start = i
        while i < n and ctx[i] != ' ': i += 1
        props.append(ctx[start:i])
    return props


def read_qa_pairs_sampled(data_path, split, max_qa):
    reservoir = []
    count = 0
    for fpath in sorted(glob.glob(os.path.join(data_path, split, '*.json'))):
        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rec = json.loads(line)
                ctx = rec['context']
                for q in rec['questions']:
                    count += 1
                    item = (ctx, q['question'], q['answers'])
                    if len(reservoir) < max_qa:
                        reservoir.append(item)
                    else:
                        j = random.randint(0, count - 1)
                        if j < max_qa: reservoir[j] = item
    print(f'  Scanned {count}, kept {len(reservoir)}')
    random.shuffle(reservoir)
    return reservoir


def read_qa_pairs_validation(data_path):
    qa = []
    for fp in sorted(glob.glob(os.path.join(data_path, 'validation', '*.json'))):
        with open(fp) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rec = json.loads(line)
                ctx = rec['context']
                for q in rec['questions']:
                    qa.append((ctx, q['question'], q['answers']))
    return qa

print('Helpers loaded.')

## Stage 1: Fine-tune BGE embedder (~2 hours)

In [ ]:
# Cell 4: Train BGE with contrastive loss on (question, correct_prop, hard_neg)
bge_save_path = f'{OUTPUT}/checkpoints/{BGE_NAME}'

if os.path.exists(os.path.join(bge_save_path, 'config_sentence_transformers.json')):
    print(f'SKIP: BGE already trained')
else:
    print('Building contrastive triples...')
    pm = read_qa_pairs_sampled(PM_PATH, 'train', BGE_TRAIN_PAIRS)
    ed = read_qa_pairs_sampled(ED_PATH, 'train', min(10_000, BGE_TRAIN_PAIRS // 10))
    qa_data = pm + ed
    del pm, ed; random.shuffle(qa_data)

    train_examples = []
    for ctx, q, ans in tqdm(qa_data, desc='Building'):
        if not ans['answer_start'] or not ans['text']:
            continue
        positive = ans['text'][0]
        all_props = precompute_props(ctx)
        negatives = [p for p in all_props if p != positive]
        if not negatives: continue
        hard_neg = random.choice(negatives)
        train_examples.append(InputExample(texts=[q, positive, hard_neg]))

    print(f'Built {len(train_examples)} triples')
    del qa_data; gc.collect()

    print(f'\nLoading {BGE_CKPT}...')
    model = SentenceTransformer(BGE_CKPT, device=device)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=BGE_BATCH)
    loss = losses.MultipleNegativesRankingLoss(model)
    warmup = int(len(train_loader) * BGE_EPOCHS * 0.06)

    print(f'Training BGE: {len(train_examples)} triples, {BGE_EPOCHS} epochs')
    t0 = time.time()
    model.fit(
        train_objectives=[(train_loader, loss)],
        epochs=BGE_EPOCHS,
        warmup_steps=warmup,
        optimizer_params={'lr': BGE_LR},
        output_path=bge_save_path,
        show_progress_bar=True,
        use_amp=True,
    )
    print(f'Done in {(time.time()-t0)/60:.1f} min. Saved to {bge_save_path}')
    del model, train_loader, train_examples
    gc.collect(); torch.cuda.empty_cache()

## Stage 2: Train reranker on top-K retrieved candidates (~2 hours)

In [ ]:
# Cell 5: Build reranker training data using BGE retrieval
reranker_cache = f'{OUTPUT}/reranker_train.npz'
reranker_path = f'{OUTPUT}/checkpoints/{RERANK_NAME}'

if os.path.exists(os.path.join(reranker_path, 'config.json')):
    print(f'SKIP: Reranker already trained')
elif os.path.exists(reranker_cache):
    print(f'Reranker cache exists ({os.path.getsize(reranker_cache)/1e6:.0f} MB)')
else:
    print('Building reranker training data (BGE retrieval + tokenize)...')

    # Load BGE for retrieval
    bge = SentenceTransformer(bge_save_path, device=device)

    pm = read_qa_pairs_sampled(PM_PATH, 'train', RERANK_TRAIN_QA)
    ed = read_qa_pairs_sampled(ED_PATH, 'train', min(6000, RERANK_TRAIN_QA // 10))
    qa_data = pm + ed
    del pm, ed; random.shuffle(qa_data)

    rerank_tok = AutoTokenizer.from_pretrained(RERANK_CKPT)
    pad_on_right = rerank_tok.padding_side == 'right'
    ctx_tok_idx = 1 if pad_on_right else 0

    all_ids, all_masks, all_starts, all_ends = [], [], [], []
    stats = {'kept': 0, 'dropped': 0}

    # Process in batches for efficiency
    BATCH_QA = 32
    for b0 in tqdm(range(0, len(qa_data), BATCH_QA), desc='Processing'):
        batch = qa_data[b0:b0+BATCH_QA]

        for ctx, q, ans in batch:
            if not ans['answer_start'] or not ans['text']:
                continue
            correct = ans['text'][0]
            all_props = precompute_props(ctx)
            if correct not in all_props:
                stats['dropped'] += 1; continue

            # Retrieve top-K via BGE
            q_emb = bge.encode([q], normalize_embeddings=True, show_progress_bar=False)
            prop_embs = bge.encode(all_props, normalize_embeddings=True,
                                    batch_size=256, show_progress_bar=False)
            sims = (prop_embs @ q_emb.T).flatten()
            top_idx = np.argsort(-sims)[:TOP_K_RETRIEVE].tolist()
            top_props = [all_props[i] for i in top_idx]

            # Force-include correct answer (teacher forcing)
            if correct not in top_props:
                top_props[-1] = correct
            random.shuffle(top_props)

            rerank_ctx = ' '.join(top_props)
            ans_start = rerank_ctx.find(correct)
            if ans_start < 0:
                stats['dropped'] += 1; continue

            tokenized = rerank_tok(
                q if pad_on_right else rerank_ctx,
                rerank_ctx if pad_on_right else q,
                truncation='only_second' if pad_on_right else 'only_first',
                max_length=MAX_LENGTH,
                padding='max_length',
                return_offsets_mapping=True,
            )
            input_ids = tokenized['input_ids']
            seq_ids = tokenized.sequence_ids()
            offsets = tokenized['offset_mapping']

            ans_end = ans_start + len(correct)
            ts, te = 0, MAX_LENGTH - 1
            while ts < MAX_LENGTH and seq_ids[ts] != ctx_tok_idx: ts += 1
            while te >= 0 and seq_ids[te] != ctx_tok_idx: te -= 1

            if ts >= MAX_LENGTH or te < 0:
                stats['dropped'] += 1; continue
            if offsets[ts][0] > ans_start or offsets[te][1] < ans_end:
                stats['dropped'] += 1; continue

            while ts < MAX_LENGTH and offsets[ts][0] <= ans_start: ts += 1
            sp = max(ts - 1, 0)
            while te >= 0 and offsets[te][1] >= ans_end: te -= 1
            ep = min(te + 1, MAX_LENGTH - 1)

            if sp > ep:
                stats['dropped'] += 1; continue

            all_ids.append(input_ids)
            all_masks.append(tokenized['attention_mask'])
            all_starts.append(sp)
            all_ends.append(ep)
            stats['kept'] += 1

    print(f'\nKept: {stats["kept"]}, Dropped: {stats["dropped"]}')
    np.savez_compressed(reranker_cache,
                        ids=np.array(all_ids, dtype=np.int32),
                        masks=np.array(all_masks, dtype=np.int32),
                        starts=np.array(all_starts, dtype=np.int64),
                        ends=np.array(all_ends, dtype=np.int64))
    print(f'Saved: {reranker_cache}')
    del qa_data, all_ids, all_masks, all_starts, all_ends, bge
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 6: Train reranker (RoBERTa-base) on retrieved contexts
if os.path.exists(os.path.join(reranker_path, 'config.json')):
    print(f'SKIP: Reranker already trained')
else:
    class NpzDataset(Dataset):
        def __init__(self, path):
            d = np.load(path)
            self.ids = d['ids'].astype(np.int64)
            self.masks = d['masks'].astype(np.int64)
            self.starts = d['starts'].astype(np.int64)
            self.ends = d['ends'].astype(np.int64)
        def __len__(self): return len(self.ids)
        def __getitem__(self, i):
            return {
                'input_ids': torch.from_numpy(self.ids[i].copy()),
                'attention_mask': torch.from_numpy(self.masks[i].copy()),
                'start_positions': torch.tensor(self.starts[i]),
                'end_positions': torch.tensor(self.ends[i]),
            }

    print(f'\nLoading {RERANK_CKPT}...')
    model = AutoModelForQuestionAnswering.from_pretrained(RERANK_CKPT)
    train_ds = NpzDataset(reranker_cache)
    print(f'Training samples: {len(train_ds)}')

    args = TrainingArguments(
        output_dir=f'{OUTPUT}/checkpoints/{RERANK_NAME}_tmp',
        per_device_train_batch_size=RERANK_BATCH,
        gradient_accumulation_steps=2,
        num_train_epochs=RERANK_EPOCHS,
        learning_rate=RERANK_LR,
        lr_scheduler_type='cosine',
        warmup_ratio=0.06,
        weight_decay=0.01,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        logging_steps=100,
        save_strategy='epoch',
        save_total_limit=1,
        report_to='none',
        dataloader_num_workers=2,
        remove_unused_columns=False,
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds)

    print('\nTraining reranker...')
    t0 = time.time()
    trainer.train()
    print(f'Done in {(time.time()-t0)/60:.1f} min')

    if os.path.exists(reranker_path):
        shutil.rmtree(reranker_path)
    trainer.model.save_pretrained(reranker_path)
    AutoTokenizer.from_pretrained(RERANK_CKPT).save_pretrained(reranker_path)
    print(f'Saved to {reranker_path}')

    del model, trainer, train_ds
    gc.collect(); torch.cuda.empty_cache()

## Stage 3: RAG evaluation (~1-2 hours)

In [ ]:
# Cell 7: Full RAG pipeline eval
print('Loading BGE + reranker...')
bge = SentenceTransformer(bge_save_path, device=device)
reranker = AutoModelForQuestionAnswering.from_pretrained(reranker_path)
reranker.to(device).half().eval()
rerank_tok = AutoTokenizer.from_pretrained(reranker_path)
print('Loaded.')

def rag_eval(data_path, ds_name, top_k=TOP_K_RETRIEVE):
    print(f'\n{"="*60}\n  RAG EVAL on {ds_name}\n{"="*60}')
    qa_pairs = read_qa_pairs_validation(data_path)
    print(f'  {len(qa_pairs)} QA pairs')

    TOP_K = list(range(1, 11))
    results = [{'rank': k, 'correct': 0, 'total': 0,
                'correct_answerable': 0, 'correct_non_answerable': 0,
                'total_answerable': 0, 'total_non_answerable': 0} for k in TOP_K]

    # Cache schema embeddings
    schema_to_emb = {}

    t0 = time.time()

    for ctx, q, ans in tqdm(qa_pairs, desc='RAG'):
        gt_name = ans['text'][0] if ans['text'] and ans['answer_start'] else None
        is_oos = gt_name is None

        if id(ctx) not in schema_to_emb:
            props = precompute_props(ctx)
            if not props:
                schema_to_emb[id(ctx)] = (props, np.zeros((0, 384)))
            else:
                emb = bge.encode(props, normalize_embeddings=True,
                                 batch_size=512, show_progress_bar=False)
                schema_to_emb[id(ctx)] = (props, emb)

        props, prop_embs = schema_to_emb[id(ctx)]

        if len(props) == 0:
            pred_ranked = [None]
        else:
            q_emb = bge.encode([q], normalize_embeddings=True, show_progress_bar=False)
            sims = (prop_embs @ q_emb.T).flatten()
            top_idx = np.argsort(-sims)[:top_k].tolist()
            top_props = [props[i] for i in top_idx]

            rerank_ctx = ' '.join(top_props)
            inputs = rerank_tok(q, rerank_ctx, truncation='only_second',
                                max_length=MAX_LENGTH, padding='max_length',
                                return_tensors='pt', return_offsets_mapping=True)
            offsets = inputs.pop('offset_mapping')[0].numpy()
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                out = reranker(**inputs)
            sl = out.start_logits[0].cpu().float().numpy()
            el = out.end_logits[0].cpu().float().numpy()

            # Enumerate candidates, match to top_props
            N_BEST = 20
            best_s = np.argsort(-sl)[:N_BEST]
            best_e = np.argsort(-el)[:N_BEST]

            candidates = []
            for si in best_s:
                si = int(si)
                if si == 0: continue
                for ei in best_e:
                    ei = int(ei)
                    if ei == 0 or ei < si or ei - si > 15: continue
                    char_s = int(offsets[si, 0])
                    char_e = int(offsets[ei, 1])
                    if char_s == 0 and char_e == 0: continue
                    pred_text = rerank_ctx[char_s:char_e].strip()
                    # Match to top_props
                    match = None
                    for p in top_props:
                        if p == pred_text:
                            match = p; break
                    if match is None:
                        for p in top_props:
                            if pred_text in p or p in pred_text:
                                match = p; break
                    if match is None: continue
                    score = float(sl[si] + el[ei])
                    candidates.append((score, match))

            # CLS candidate (no answer)
            candidates.append((float(sl[0] + el[0]), None))
            candidates.sort(key=lambda x: -x[0])

            pred_ranked = []
            seen = set()
            for _, name in candidates:
                key = name if name is not None else '__NULL__'
                if key in seen: continue
                seen.add(key)
                pred_ranked.append(name)
                if len(pred_ranked) >= 10: break

        rank = None
        for ri, pred in enumerate(pred_ranked):
            if is_oos:
                if pred is None: rank = ri + 1; break
            else:
                if pred == gt_name: rank = ri + 1; break

        for kr in results:
            if rank is not None and rank <= kr['rank']:
                kr['correct'] += 1
                if is_oos: kr['correct_non_answerable'] += 1
                else: kr['correct_answerable'] += 1
            kr['total'] += 1
            if is_oos: kr['total_non_answerable'] += 1
            else: kr['total_answerable'] += 1

    for kr in results:
        t = kr['total']; ta = kr['total_answerable']; tn = kr['total_non_answerable']
        kr['accuracy'] = kr['correct']/t if t else 0
        kr['accuracy_answerable'] = kr['correct_answerable']/ta if ta else 0
        kr['accuracy_non_answerable'] = kr['correct_non_answerable']/tn if tn else 0

    elapsed = time.time() - t0
    print(f'\n  Done: {elapsed/60:.1f} min')
    print(f'  Ans: {results[0]["total_answerable"]} | Non-ans: {results[0]["total_non_answerable"]}')
    print(f'  K   Acc@K   Ans     Non')
    for r in results:
        print(f"  @{r['rank']:>2}  {r['accuracy']:.4f}  {r['accuracy_answerable']:.4f}  {r['accuracy_non_answerable']:.4f}")

    return {
        'model': f'RAG_BGE+RoBERTa', 'dataset': ds_name, 'type': 'rag',
        'total_samples': results[0]['total'],
        'total_answerable': results[0]['total_answerable'],
        'total_non_answerable': results[0]['total_non_answerable'],
        'elapsed_seconds': elapsed, 'results': results,
    }


all_results = []
r = rag_eval(PM_PATH, 'parameter_matching')
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
with open(f'{OUTPUT}/results/rag_pm_{ts}.json', 'w') as f:
    json.dump(r, f, indent=2)
all_results.append(r)

r = rag_eval(ED_PATH, 'endpoint_discovery')
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
with open(f'{OUTPUT}/results/rag_ed_{ts}.json', 'w') as f:
    json.dump(r, f, indent=2)
all_results.append(r)

print('\n✓ Eval done!')

In [ ]:
# Cell 8: Final comparison
baseline = [
    {'model': 'CB-PM+ED (paper best)', 'dataset': 'parameter_matching', 'type': 'baseline',
     'results': [{'rank': 1, 'accuracy': 0.8208}, {'rank': 3, 'accuracy': 0.9613},
                 {'rank': 5, 'accuracy': 0.9759}, {'rank': 10, 'accuracy': 0.9816}]},
    {'model': 'CB-ED (paper)', 'dataset': 'endpoint_discovery', 'type': 'baseline',
     'results': [{'rank': 1, 'accuracy': 0.8844}, {'rank': 3, 'accuracy': 0.9535},
                 {'rank': 5, 'accuracy': 0.9678}, {'rank': 10, 'accuracy': 0.9749}]},
]

combined = baseline + all_results

print(f'\n{"="*100}')
print(f'RAG vs paper — FINAL')
print(f'{"="*100}')
print(f'{"Model":<30} {"Type":<10} {"Dataset":<22} {"Acc@1":>7} {"Acc@3":>7} {"Acc@5":>7} {"Acc@10":>7}')
print('-'*100)
for r in combined:
    acc = {x['rank']: x.get('accuracy', 0) for x in r['results']}
    print(f'{r["model"]:<30} {r["type"]:<10} {r["dataset"]:<22} '
          f'{acc.get(1,0):>7.4f} {acc.get(3,0):>7.4f} {acc.get(5,0):>7.4f} {acc.get(10,0):>7.4f}')
print('='*100)

for ds in ['parameter_matching', 'endpoint_discovery']:
    paper = max([{x['rank']: x['accuracy'] for x in r['results']}.get(1, 0)
                 for r in baseline if r['dataset'] == ds] or [0])
    ours = max([{x['rank']: x['accuracy'] for x in r['results']}.get(1, 0)
                for r in all_results if r['dataset'] == ds] or [0])
    diff = (ours - paper) * 100
    sign = '🎯' if diff > 0 else '⚠'
    print(f'\n{sign} {ds}: RAG {ours:.4f} vs paper {paper:.4f} → {diff:+.2f}%')

with open(f'{OUTPUT}/results/rag_final_comparison.json', 'w') as f:
    json.dump(combined, f, indent=2)